In [ ]:
# -*- coding: utf-8 -*-
###########블로그 분석기###########

"""
CSV 입력 기반 감성어/형용사 분석 스크립트
- KoNLPy 불필요, 경량 한국어 토크나이저 사용 (조사/어미/접미 정리 + 기본형 근사)
- KnuSentiLex(JSON/CSV/TSV) 자동 파싱
- 결과: 표 출력 + CSV 저장(+ 선택 워드클라우드 저장)

사용법 예)
run_csv(
    csv_path="gln_blog_data.csv",
    text_col="content",
    out_dir="csv_out",
    make_wordclouds=True
)
"""

import os, re, csv, json, math
from pathlib import Path
from typing import List, Tuple, Optional
from collections import Counter

import pandas as pd
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# ===================== 사용자 설정 =====================
# KnuSentiLex 경로(환경변수 KNU_LEXICON_PATH 우선)
DEFAULT_KNU_CANDIDATES = [
    os.path.expanduser("/Users/jiyeonkim/GLN/KnuSentiLex-master/data/SentiWord_info.json"), # SentiWord_info.json 경로
    os.path.expanduser("~/KnuSentiLex/data/SentiWord_info.json"), # SentiWord_info.json 경로
]
KNU_LEXICON_PATH = os.environ.get("KNU_LEXICON_PATH", "")  # 없으면 아래 candidates 탐색

# 워드클라우드 색상
COLOR_POS = "#2ECC71"
COLOR_NEG = "#E74C3C"
COLOR_NEU = "#7F8C8D"

# 중요 토큰 선별 파라미터 (워드클라우드용)
TOP_K = 500
MIN_DF = 2
MAX_DF_RATIO = 0.7
MIN_LEN = 2

# ===================== 폰트 탐색 (Mac 기준, 필요시 수정) =====================
def find_korean_font() -> Optional[str]:
    candidates = [
        "/System/Library/Fonts/AppleSDGothicNeo.ttc",
        "/System/Library/Fonts/Supplemental/AppleGothic.ttf",
        "/Library/Fonts/NanumGothic.ttf",
        "/Library/Fonts/NanumGothic.otf",
        "/Library/Fonts/AppleGothic.ttf",
        os.path.expanduser("~/Library/Fonts/NanumGothic.ttf"),
    ]
    for p in candidates:
        if os.path.isfile(p):
            return p
    return None

# ===================== 전처리 & 경량 형태소 토큰화 =====================
_hangul = re.compile(r"[^\uAC00-\uD7A3a-zA-Z0-9\s]")
_space = re.compile(r"\s+")
_num = re.compile(r"^\d+$")

BASE_STOPWORDS = """
은 는 이 가 을 를 와 과 도 만 에 에서 으로 로 보다 까지 하고 또는 그리고 그러나 또한 및 등 등등
대한 대해 대해서 위해 위하여 통해 관련 라는 같은 등의 보다에서보다 대한것 것들 것 이나 또는 어떤 경우 등에 의해 위해서 로서 로써 보다도 에게 에서는 에서도 에도 의 과의 와의
합니다 한다 했다 하하였다 하는 하다 되다 되어 되었다 되게 및과 및과의 및으로
""".split()
CUSTOM_STOPWORDS = set()

JOSA_ENDINGS = ("은","는","이","가","을","를","와","과","도","만","에","에서","으로","로","에게","께","부터","까지","보다","처럼","같이")
EOMI_ENDINGS = ("습니다","니다","어요","아요","였어요","했다","한다","하는","했다면","하며","하여","해서","하면","하니","했다가","했다며","했다는","합니다","하였다","했다네요","했네요")
SUFFIXES = ("적","성","화","들","께","께서","씨","님")

def normalize_text(t: str) -> str:
    t = re.sub(r"http[s]?://\S+", " ", t)
    t = _hangul.sub(" ", t)
    t = _space.sub(" ", t).strip()
    return t

def _strip_trailing(token: str, endings: tuple) -> str:
    for e in endings:
        if token.endswith(e) and len(token) > len(e)+1:
            return token[: -len(e)]
    return token

def _to_base_verb_adj(token: str) -> str:
    """하다/되다 계열 간단 정규화 + 어미 제거 후 '다' 부착(근사)"""
    if token.endswith(("합니다","하였다","한다","해요","했어요","하는","하면","하여","해서","했다","했다가")):
        return "하다"
    if token.endswith(("됩니다","되었다","된다","돼요","됐어요","되는","되면","되어","돼서","됐다")):
        return "되다"
    base = _strip_trailing(token, EOMI_ENDINGS)
    if base and base[-1] != "다" and len(base) >= 2 and base != token:
        return base + "다"
    return token

def tokenize_krlite(text: str) -> List[str]:
    """조사/접미/어미 제거 + 기본형 근사"""
    text = normalize_text(text)
    raw = text.split()
    toks = []
    for w in raw:
        if len(w) < 2 or _num.match(w) or w in BASE_STOPWORDS or w in CUSTOM_STOPWORDS:
            continue
        w1 = _strip_trailing(w, JOSA_ENDINGS)
        w1 = _strip_trailing(w1, SUFFIXES)
        w2 = _to_base_verb_adj(w1)
        if len(w2) >= 2 and not _num.match(w2) and w2 not in BASE_STOPWORDS and w2 not in CUSTOM_STOPWORDS:
            toks.append(w2)
    return toks

# ===================== KnuSentiLex 로더 =====================
def load_knu_lexicon(path: Optional[str] = None) -> dict:
    # 경로 우선순위: 인자 → 환경변수 → candidates
    cand = []
    if path: cand.append(path)
    if KNU_LEXICON_PATH: cand.append(KNU_LEXICON_PATH)
    cand.extend(DEFAULT_KNU_CANDIDATES)
    path_use = None
    for p in cand:
        if p and os.path.isfile(p):
            path_use = p
            break
    if not path_use:
        raise FileNotFoundError("KnuSentiLex 파일을 찾을 수 없습니다. KNU_LEXICON_PATH 또는 DEFAULT_KNU_CANDIDATES 확인.")

    ext = os.path.splitext(path_use)[1].lower()
    lex = {}

    def map_pol(v):
        if v is None:
            return 0
        s = str(v).strip().lower()
        if s in ("pos","positive","1","2","+1","+2"):
            return 1
        if s in ("neg","negative","-1","-2"):
            return -1
        if s in ("neu","neutral","0"):
            return 0
        try:
            n = int(s)
            return 1 if n > 0 else (-1 if n < 0 else 0)
        except Exception:
            return 0

    if ext == ".json":
        with open(path_use, "r", encoding="utf-8") as f:
            data = json.load(f)
        # JSON 구조 가변성 대응
        for item in data:
            w = item.get("word") or item.get("lemma") or item.get("term") or item.get("entry")
            pol = item.get("polarity") or item.get("pol") or item.get("label") or item.get("value")
            if w:
                lex[w.strip()] = map_pol(pol)
    else:
        # CSV/TSV/기타 구분자 자동 탐지
        with open(path_use, "r", encoding="utf-8") as fh:
            head = fh.read(2048)
            sep = "\t" if "\t" in head else ("," if "," in head else None)
        candidates = [sep] if sep else []
        for s in [",", "\t", ";", "|", " "]:
            if s not in candidates:
                candidates.append(s)
        loaded = False
        for s in candidates:
            try:
                with open(path_use, "r", encoding="utf-8") as fh:
                    reader = csv.reader(fh, delimiter=s)
                    cnt = 0
                    for row in reader:
                        if not row:
                            continue
                        # 헤더 스킵
                        if cnt == 0 and any(h.lower() in ("word","lemma","term","entry","polarity","pol","label","value") for h in row):
                            cnt += 1
                            continue
                        if len(row) == 1:
                            parts = row[0].split()
                            if len(parts) >= 2:
                                w, pol = parts[0], parts[1]
                            else:
                                continue
                        else:
                            w, pol = row[0], row[1] if len(row) > 1 else "0"
                        if w:
                            lex[w.strip()] = map_pol(pol)
                        cnt += 1
                loaded = True
                break
            except Exception:
                continue
        if not loaded:
            raise ValueError("KnuSentiLex 파일 파싱 실패")

    print(f"[KNU] 로딩 완료: {len(lex)}개 단어 | {path_use}")
    return lex

# ===================== 형용사(heuristic) 추출 =====================
# 아주 간단한 근사 규칙: '다'로 끝나면서 흔한 동사/보조용언 기본형을 제외
VERB_EXCLUDES = {"하다","되다","있다","없다","먹다","가다","오다","보다","알다","말하다","주다","받다","살다","보다","만나다","얻다","넣다","놓다"}

def is_adj_like(token: str) -> bool:
    """형용사 근사 판단: '다'로 끝나고, 자주 쓰는 동사 기본형 제외"""
    if len(token) < 2: 
        return False
    if not token.endswith("다"):
        return False
    if token in VERB_EXCLUDES:
        return False
    # 어간 끝이 '하/되'로 끝나면 동사일 가능성 ↑ → 제외
    stem = token[:-1]
    if len(stem) >= 1 and stem.endswith(("하","되")):
        return False
    return True

# ===================== 중요 토큰 선별 (워드클라우드용) =====================
def select_important_tokens(docs_tokens: List[List[str]], top_k=TOP_K, min_df=MIN_DF, max_df_ratio=MAX_DF_RATIO, min_len=MIN_LEN):
    N = len(docs_tokens) if docs_tokens else 1
    df = Counter()
    for toks in docs_tokens:
        df.update(set([w for w in toks if len(w) >= min_len]))
    vocab = {w for w, c in df.items() if c >= min_df and (c / max(1, N)) <= max_df_ratio} or set(df.keys())
    tf = Counter()
    for toks in docs_tokens:
        for w in toks:
            if w in vocab:
                tf[w] += 1
    scores = {}
    for w in vocab:
        idf = math.log((N + 1) / (df[w] + 1)) + 1.0
        scores[w] = tf[w] * idf
    important = {w for w, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]}
    return important

# ===================== 워드클라우드 =====================
def make_colored_wordcloud(freq_map: dict, side_map: dict, out_path: str, max_words=800):
    if not freq_map:
        print("[워클] 표시할 토큰이 없습니다:", out_path)
        return
    font = find_korean_font()
    wc = WordCloud(
        font_path=font,
        width=2000,
        height=1200,
        background_color="white",
        max_words=max_words,
    ).generate_from_frequencies(freq_map)

    def color_by_side(word, *args, **kwargs):
        lab = side_map.get(word, "NEU")
        if lab == "POS": return COLOR_POS
        if lab == "NEG": return COLOR_NEG
        return COLOR_NEU

    wc = wc.recolor(color_func=color_by_side)
    plt.figure(figsize=(20, 12))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(out_path, dpi=180)
    plt.close()
    print("[저장]", out_path)

# ===================== 감성 라벨 =====================
def label_token(token: str, lexicon: dict) -> str:
    pol = lexicon.get(token)
    if pol is None:
        return "UNK"
    return "POS" if pol > 0 else ("NEG" if pol < 0 else "NEU")

# ===================== CSV 실행 진입점 =====================
def read_csv_robust(path: str) -> pd.DataFrame:
    last_err = None
    for enc in ["utf-8-sig", "utf-8", "cp949"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception as e:
            last_err = e
    raise RuntimeError(f"CSV 읽기 실패(utf-8-sig/utf-8/cp949). 마지막 오류: {last_err}")

def run_csv(
    csv_path: str,
    text_col: str = "content",
    out_dir: str = "csv_out",
    make_wordclouds: bool = True
):
    os.makedirs(out_dir, exist_ok=True)

    # 1) KnuSentiLex 로드
    lexicon = load_knu_lexicon()

    # 2) CSV 로드
    df = read_csv_robust(csv_path)
    if text_col not in df.columns:
        raise ValueError(f"'{text_col}' 열이 없습니다. 현재 열: {list(df.columns)}")

    texts = df[text_col].dropna().astype(str).tolist()

    # 3) 토큰화
    docs_tokens = [tokenize_krlite(t) for t in texts]
    all_tokens = [w for toks in docs_tokens for w in toks]

    # 4) 감성어 빈도
    senti_freq = {"POS": Counter(), "NEG": Counter(), "NEU": Counter(), "UNK": Counter()}
    for w, c in Counter(all_tokens).items():
        lab = label_token(w, lexicon)
        senti_freq[lab][w] = c

    # 5) 형용사 빈도(heuristic)
    adj_tokens = [w for w in all_tokens if is_adj_like(w)]
    adj_freq = Counter(adj_tokens)

    # 6) 표 출력 (상위 n개)
    top_n = 30
    def df_top(counter: Counter, name: str):
        items = counter.most_common(top_n)
        d = pd.DataFrame(items, columns=[name, "freq"])
        return d

    pos_df = df_top(senti_freq["POS"], "token(POS)")
    neg_df = df_top(senti_freq["NEG"], "token(NEG)")
    neu_df = df_top(senti_freq["NEU"], "token(NEU)")
    unk_df = df_top(senti_freq["UNK"], "token(UNK)")
    adj_df = df_top(adj_freq, "adjective_like")

    print("\n📊 감성어 Top 30 (POS)")
    print(pos_df.to_string(index=False))
    print("\n📉 감성어 Top 30 (NEG)")
    print(neg_df.to_string(index=False))
    print("\n😐 감성어 Top 30 (NEU/사전에 존재)")
    print(neu_df.to_string(index=False))
    print("\n❔ 사전에 없는 토큰 Top 30 (UNK)")
    print(unk_df.to_string(index=False))
    print("\n🔤 형용사(근사) Top 30")
    print(adj_df.to_string(index=False))

    # 7) 전체 결과 저장
    # 7-1) 감성어 전체
    rows = []
    for lab, ctr in senti_freq.items():
        for w, c in ctr.most_common():
            rows.append((w, c, lab))
    senti_all_df = pd.DataFrame(rows, columns=["token", "freq", "polarity"])
    senti_csv = os.path.join(out_dir, "sentiment_tokens.csv")
    senti_all_df.to_csv(senti_csv, index=False, encoding="utf-8-sig")
    print("[저장]", senti_csv)

    # 7-2) 형용사 전체
    adj_all_df = pd.DataFrame(adj_freq.most_common(), columns=["adjective_like", "freq"])
    adj_csv = os.path.join(out_dir, "adjectives.csv")
    adj_all_df.to_csv(adj_csv, index=False, encoding="utf-8-sig")
    print("[저장]", adj_csv)

    # 8) 워드클라우드(선택)
    if make_wordclouds:
        imp = select_important_tokens(docs_tokens, top_k=TOP_K, min_df=MIN_DF, max_df_ratio=MAX_DF_RATIO, min_len=MIN_LEN)
        all_freq = Counter(all_tokens)
        freq_map = {w: c for w, c in all_freq.items() if w in imp}

        side_map = {}
        for w in freq_map.keys():
            lab = label_token(w, lexicon)
            side_map[w] = "NEU" if lab == "UNK" else lab

        wc_path = os.path.join(out_dir, "wordcloud_sentiment.png")
        make_colored_wordcloud(freq_map, side_map, wc_path, max_words=min(TOP_K, 1000))

    print("\n✅ 완료! 출력 폴더:", os.path.abspath(out_dir))
    print(" - sentiment_tokens.csv (토큰, 빈도, 극성)") ## 감성어 아웃풋
    print(" - adjectives.csv (형용사 근사, 빈도)") ##형용사 아웃풋
    if make_wordclouds:
        print(" - wordcloud_sentiment.png (극성 색상)") ##감성어 워드클라우드 파일 아웃풋

# --------------------- 실행 예시 ---------------------
run_csv(
    csv_path="/Users/jiyeonkim/GLN/gln_blog_data.csv",    ##블로그 csv 파일 경로(크롤링한 자료 경로)
    text_col="content",
    out_dir="csv_out",
    make_wordclouds=True
)

In [ ]:
# -*- coding: utf-8 -*-

##########카페 분석기###########

"""
CSV → 감성어/형용사 분석 (비문 제거판)
- KoNLPy 불필요: 조사/접미/어미 제거 + '용언에 한해' 기본형 근사
- 명사에 '다'를 붙이지 않음
- 형용사 판정: 코어/화이트리스트/형용사형 접미 패턴 중심
- KnuSentiLex(JSON/CSV/TSV) 자동 로드
"""

import os, re, csv, json, math
from typing import List, Optional, Dict
from collections import Counter
from pathlib import Path

import pandas as pd
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# ====== 사용자 설정 ======
DEFAULT_KNU_CANDIDATES = [
    os.path.expanduser("/Users/jiyeonkim/GLN/KnuSentiLex-master/data/SentiWord_info.json"), # SentiWord_info.json 경로
    os.path.expanduser("~/KnuSentiLex/data/SentiWord_info.json"), # SentiWord_info.json 경로
]
KNU_LEXICON_PATH = os.environ.get("KNU_LEXICON_PATH", "")

COLOR_POS = "#2ECC71"
COLOR_NEG = "#E74C3C"
COLOR_NEU = "#7F8C8D"

TOP_K = 500
MIN_DF = 2
MAX_DF_RATIO = 0.7
MIN_LEN = 2

# ====== 폰트 ======
def find_korean_font() -> Optional[str]:
    for p in [
        "/System/Library/Fonts/AppleSDGothicNeo.ttc",
        "/System/Library/Fonts/Supplemental/AppleGothic.ttf",
        "/Library/Fonts/NanumGothic.ttf",
        "/Library/Fonts/NanumGothic.otf",
        "/Library/Fonts/AppleGothic.ttf",
        os.path.expanduser("~/Library/Fonts/NanumGothic.ttf"),
    ]:
        if os.path.isfile(p):
            return p
    return None

# ====== 전처리 & 토크나이즈 ======
_hangul = re.compile(r"[^\uAC00-\uD7A3a-zA-Z0-9\s]")
_space  = re.compile(r"\s+")
_num    = re.compile(r"^\d+$")
_has_latin = re.compile(r"[A-Za-z]")

BASE_STOPWORDS = """
은 는 이 가 을 를 와 과 도 만 에 에서 으로 로 보다 까지 하고 또는 그리고 그러나 또한 및 등 등등
대한 대해 대해서 위해 위하여 통해 관련 라는 같은 등의 대한것 것들 것 이나 어떤 경우 등에 의해 위해서 로서 로써 에게 에서는 에서도 에도 의
합니다 한다 했다 하였다 하는 하다 되다 되어 되었다 되게
""".split()
CUSTOM_STOPWORDS: set = set()

JOSA_ENDINGS = ("은","는","이","가","을","를","와","과","도","만","에","에서","으로","로","에게","께","부터","까지","보다","처럼","같이")
SUFFIXES     = ("적","성","화","들","께","께서","씨","님")

def normalize_text(t: str) -> str:
    t = re.sub(r"http[s]?://\S+", " ", t)
    t = _hangul.sub(" ", t)
    t = _space.sub(" ", t).strip()
    return t

def _strip_trailing(token: str, endings: tuple) -> str:
    for e in endings:
        if token.endswith(e) and len(token) > len(e)+1:
            return token[:-len(e)]
    return token

# --- 용언만 기본형으로: 명사에 '다' 절대 붙이지 않음 ---
# 용언 어미/정중체/과거형 탐지
_RE_POLITE  = re.compile(r"(습니다|습니까|읍니다|이에요|예요|어요|아요)$")
_RE_PAST_EO = re.compile(r"(었어요|았어요|였어요)$")
_RE_PAST_DA = re.compile(r"(었|았|였)다$")
_RE_DO      = re.compile(r"(해요|했어요|합니다|한다)$")
_RE_BECOME  = re.compile(r"(돼요|됐어요|됩니다|된다)$")
_RE_COPULA  = re.compile(r"(입니다|이다)$")

def to_lemma(token: str) -> str:
    """
    - 하다/되다/있다/없다/이다 계열만 안정적으로 표제형화
    - 그 외엔 '다' 미부착 (명사 보존)
    """
    w = token

    # 있다/없다 계열 (있어요/없어요 → 있다/없다)
    if w.startswith("있") or w.startswith("없"):
        w = re.sub(_RE_PAST_EO, "다", w)
        w = re.sub(_RE_PAST_DA, "다", w)
        w = re.sub(_RE_POLITE, "다", w)
        return w if w.endswith("다") else w

    # 하다/되다
    if _RE_DO.search(w):
        return "하다"
    if _RE_BECOME.search(w):
        return "되다"
    if w.endswith(("했다","하여","해서","하면","하는")):
        return "하다"
    if w.endswith(("됐다","되어","되면","되는")):
        return "되다"

    # 과거형 표준화: ~았다/었다/였다 → ~다 (용언에 한함: 이미 '다'로 끝나는 경우만)
    if _RE_PAST_DA.search(w):
        return _RE_PAST_DA.sub("다", w)

    # 정중체: ~습니다/어요/아요 → ~다 (용언에 한함: 이미 용언으로 보일 때만)
    if _RE_POLITE.search(w) and not _has_latin.search(w):
        base = _RE_POLITE.sub("다", w)
        return base

    # 서술격 조사 '이다'는 그대로 두기(형용사로 보지 않음)
    if _RE_COPULA.search(w):
        return _RE_COPULA.sub("이다", w)

    # 기본: 그대로 (명사 보존)
    return w

def tokenize_krlite(text: str) -> List[str]:
    text = normalize_text(text)
    out = []
    for w in text.split():
        if len(w) < 2 or _num.match(w):
            continue
        if w in BASE_STOPWORDS or w in CUSTOM_STOPWORDS:
            continue
        w = _strip_trailing(w, JOSA_ENDINGS)
        w = _strip_trailing(w, SUFFIXES)
        w = to_lemma(w)
        # 라틴 포함 토큰은 그대로 두되, 형용사 판정에서 걸러짐
        if len(w) >= 2 and not _num.match(w) and w not in BASE_STOPWORDS:
            out.append(w)
    return out

# ====== 형용사 판정 ======
HADA_ADJ_BASES = {
    "가능","필요","중요","간단","복잡","편리","불편","친절","정확","신속","안전",
    "유용","유익","적절","부적절","합리","효율","효과","쾌적","명확","부정확","필수","청결"
}
VERB_BASES = {
    "가다","오다","먹다","마시다","하다","되다","주다","드리다","받다","만나다","보다","얻다",
    "넣다","놓다","구매하다","결제하다","출금하다","추천드리다","추천하다","사용하다","확인하다",
    "나오다","다녀오다","가입하다","클릭하다","등록하다","충전하다","환전하다"
}
ADJ_WHITELIST = {
    "좋다","나쁘다","많다","적다","같다","다르다","쉽다","어렵다","빠르다","느리다",
    "높다","낮다","크다","작다","길다","짧다","가깝다","멀다","뜨겁다","차갑다",
    "깨끗하다","더럽다","편하다","불편하다","친절하다","신속하다","정확하다","안전하다",
    "행복하다","기쁘다","슬프다","싫다","좋아하다","맛있다","재미있다","유용하다","필요하다",
    "중요하다","가능하다","명확하다","부정확하다","간단하다","복잡하다","편리하다"
}
ADJ_CORE = {"있다","없다"}

def is_adjective(token: str) -> bool:
    # 라틴/숫자 포함은 형용사 아님
    if _has_latin.search(token) or any(ch.isdigit() for ch in token):
        return False

    if token in ADJ_CORE or token in ADJ_WHITELIST:
        return True

    # 하다형 형용사
    if token.endswith("하다"):
        stem = token[:-2]
        return stem in HADA_ADJ_BASES

    # 전형적 형용사 접미
    if token.endswith(("스럽다","답다","롭다")):
        return True

    # '…있다' (맛있다/재미있다 등)
    if token.endswith("있다") and len(token) > 2:
        return True

    # 동사/행위류 제외
    if token in VERB_BASES:
        return False

    # 기본: 보수적으로 False
    return False

# 흔한 비자연형 보정 (오타 교정)
FIX_MAP = {
    "있습다":"있다","좋습다":"좋다","같습다":"같다","했습다":"하다",
    "추천드립다":"추천드리다","가능다":"가능하다","바랍다":"바라다",
    "있으니다":"있습니다"
}

# ====== KnuSentiLex ======
def load_knu_lexicon(path: Optional[str] = None) -> Dict[str, int]:
    cand = []
    if path: cand.append(path)
    if KNU_LEXICON_PATH: cand.append(KNU_LEXICON_PATH)
    cand.extend(DEFAULT_KNU_CANDIDATES)
    use = None
    for p in cand:
        if p and os.path.isfile(p):
            use = p; break
    if not use:
        raise FileNotFoundError("KnuSentiLex 경로를 확인하세요.")

    def map_pol(v):
        if v is None: return 0
        s = str(v).strip().lower()
        if s in ("pos","positive","1","2","+1","+2"): return 1
        if s in ("neg","negative","-1","-2"):        return -1
        if s in ("neu","neutral","0"):               return 0
        try:
            n = int(s); return 1 if n>0 else (-1 if n<0 else 0)
        except: return 0

    lex = {}
    ext = os.path.splitext(use)[1].lower()
    if ext == ".json":
        with open(use, "r", encoding="utf-8") as f:
            data = json.load(f)
        for item in data:
            w = item.get("word") or item.get("lemma") or item.get("term") or item.get("entry")
            pol = item.get("polarity") or item.get("pol") or item.get("label") or item.get("value")
            if w: lex[w.strip()] = map_pol(pol)
    else:
        with open(use, "r", encoding="utf-8") as fh:
            head = fh.read(2048)
            sep = "\t" if "\t" in head else ("," if "," in head else None)
        candidates = [sep] if sep else []
        for s in [",", "\t", ";", "|", " "]:
            if s not in candidates: candidates.append(s)
        loaded = False
        for s in candidates:
            try:
                with open(use, "r", encoding="utf-8") as fh:
                    reader = csv.reader(fh, delimiter=s)
                    cnt = 0
                    for row in reader:
                        if not row: continue
                        if cnt == 0 and any(h.lower() in ("word","lemma","term","entry","polarity","pol","label","value") for h in row):
                            cnt += 1; continue
                        if len(row) == 1:
                            parts = row[0].split()
                            if len(parts) >= 2: w, pol = parts[0], parts[1]
                            else:               continue
                        else:
                            w, pol = row[0], row[1] if len(row) > 1 else "0"
                        if w: lex[w.strip()] = map_pol(pol)
                        cnt += 1
                loaded = True; break
            except Exception:
                continue
        if not loaded:
            raise ValueError("KnuSentiLex 파일 파싱 실패")
    print(f"[KNU] 로딩: {len(lex)}개 | {use}")
    return lex

# ====== 워드클라우드 ======
def make_colored_wordcloud(freq_map: dict, side_map: dict, out_path: str, max_words=800):
    if not freq_map: return
    font = find_korean_font()
    wc = WordCloud(font_path=font, width=2000, height=1200, background_color="white",
                   max_words=max_words).generate_from_frequencies(freq_map)
    def color_by_side(word, *args, **kwargs):
        lab = side_map.get(word, "NEU")
        if lab == "POS": return COLOR_POS
        if lab == "NEG": return COLOR_NEG
        return COLOR_NEU
    wc = wc.recolor(color_func=color_by_side)
    plt.figure(figsize=(20,12))
    plt.imshow(wc, interpolation="bilinear"); plt.axis("off")
    plt.tight_layout(); plt.savefig(out_path, dpi=180); plt.close()
    print("[저장]", out_path)

# ====== 공용 ======
def read_csv_robust(path: str) -> pd.DataFrame:
    last_err = None
    for enc in ("utf-8-sig","utf-8","cp949"):
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception as e:
            last_err = e
    raise RuntimeError(f"CSV 읽기 실패. 마지막 오류: {last_err}")

def select_important_tokens(docs_tokens: List[List[str]], top_k=TOP_K, min_df=MIN_DF, max_df_ratio=MAX_DF_RATIO, min_len=MIN_LEN):
    N = len(docs_tokens) if docs_tokens else 1
    df = Counter()
    for toks in docs_tokens:
        df.update(set([w for w in toks if len(w) >= min_len]))
    vocab = {w for w, c in df.items() if c >= min_df and (c / max(1, N)) <= max_df_ratio} or set(df.keys())
    tf = Counter()
    for toks in docs_tokens:
        for w in toks:
            if w in vocab: tf[w] += 1
    scores = {w: tf[w] * (math.log((N + 1) / (df[w] + 1)) + 1.0) for w in vocab}
    return {w for w, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]}

def label_token(token: str, lexicon: Dict[str, int]) -> str:
    pol = lexicon.get(token)
    if pol is None: return "UNK"
    return "POS" if pol > 0 else ("NEG" if pol < 0 else "NEU")

# ====== 메인 ======
def run_csv(
    csv_path: str,
    text_col: str = "content",
    out_dir: str = "csv_out",
    top_n: int = 50,
    make_wordclouds: bool = True
):
    os.makedirs(out_dir, exist_ok=True)

    lexicon = load_knu_lexicon()
    df = read_csv_robust(csv_path)
    if text_col not in df.columns:
        raise ValueError(f"'{text_col}' 열이 없습니다. 현재 열: {list(df.columns)}")

    texts = df[text_col].dropna().astype(str).tolist()
    docs_tokens = [tokenize_krlite(t) for t in texts]
    all_tokens = [w for toks in docs_tokens for w in toks]

    # 감성어 빈도
    senti_freq = {"POS": Counter(), "NEG": Counter(), "NEU": Counter(), "UNK": Counter()}
    for w, c in Counter(all_tokens).items():
        lab = label_token(w, lexicon)
        senti_freq[lab][w] = c

    # 형용사 빈도 (보정 + 판정)
    adjs = []
    for w in all_tokens:
        w = FIX_MAP.get(w, w)
        if is_adjective(w):
            adjs.append(w)
    adj_freq = Counter(adjs)

    # 표 출력
    def df_top(counter: Counter, name: str, n: int):
        return pd.DataFrame(counter.most_common(n), columns=[name, "freq"])

    pos_df = df_top(senti_freq["POS"], "token(POS)", top_n)
    neg_df = df_top(senti_freq["NEG"], "token(NEG)", top_n)
    neu_df = df_top(senti_freq["NEU"], "token(NEU)", top_n)
    unk_df = df_top(senti_freq["UNK"], "token(UNK)", top_n)
    adj_df = df_top(adj_freq, "adjective", top_n)

    print("\n📊 감성어 Top", top_n, "(POS)"); print(pos_df.to_string(index=False))
    print("\n📉 감성어 Top", top_n, "(NEG)"); print(neg_df.to_string(index=False))
    print("\n😐 감성어 Top", top_n, "(NEU/사전존재)"); print(neu_df.to_string(index=False))
    print("\n❔ 사전에 없는 토큰 Top", top_n, "(UNK)"); print(unk_df.to_string(index=False))
    print("\n🔤 형용사 Top", top_n); print(adj_df.to_string(index=False))

    # 저장
    senti_rows = []
    for lab, ctr in senti_freq.items():
        for w, c in ctr.most_common():
            senti_rows.append((w, c, lab))
    pd.DataFrame(senti_rows, columns=["token","freq","polarity"])\
        .to_csv(os.path.join(out_dir,"sentiment_tokens.csv"), index=False, encoding="utf-8-sig")

    pd.DataFrame(adj_freq.most_common(), columns=["adjective","freq"])\
        .to_csv(os.path.join(out_dir,"adjectives.csv"), index=False, encoding="utf-8-sig")

    if make_wordclouds:
        imp = select_important_tokens(docs_tokens, top_k=TOP_K, min_df=MIN_DF, max_df_ratio=MAX_DF_RATIO, min_len=MIN_LEN)
        all_freq = Counter(all_tokens)
        freq_map = {w: c for w, c in all_freq.items() if w in imp}
        side_map = {w: ("NEU" if label_token(w, lexicon) == "UNK" else label_token(w, lexicon)) for w in freq_map}
        make_colored_wordcloud(freq_map, side_map, os.path.join(out_dir,"wordcloud_sentiment.png"))

    print("\n✅ 완료! 출력 폴더:", os.path.abspath(out_dir))

# ---------------- 사용 예시 ----------------
run_csv(
    csv_path="/Users/jiyeonkim/GLN/gln_cafe_data.csv",  ##카페 csv 파일 경로(크롤링한 자료 경로)
    text_col="content",
    out_dir="csv_out",
    top_n=50,
    make_wordclouds=True
)